# 第 13–14 天 · 阶段项目：命令行助手

> 📓 **这是第 13–14 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/13)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai pydantic python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 13–14 天 · 阶段项目：命令行助手（多轮对话 + 流式输出 + 3 个工具）
没有 Key 也能跑（假模型按剧本流式输出、调用工具）；填了 LLM_API_KEY 就调用真模型。
浏览器里是单文件版；本地请按课程把三段拆成 llm.py / tools.py / main.py。
新知识点：流式输出时 tool_calls 也是"一块一块"来的，要按 index 拼起来。"""
import ast
import asyncio
import json
import operator
import os
import pathlib
from types import SimpleNamespace as NS

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator

load_dotenv()

# ======================= tools.py：工具注册 =======================
REGISTRY: dict[str, tuple] = {}  # name -> (参数模型, 函数, 超时秒数)
TOOLS: list[dict] = []


def tool(args_model, description: str, timeout: float = 2.0):
    """注册工具的装饰器（类比 Spring 的 @Component + 自动扫描）"""
    def deco(func):
        REGISTRY[func.__name__] = (args_model, func, timeout)
        TOOLS.append({"type": "function", "function": {
            "name": func.__name__, "description": description,
            "parameters": args_model.model_json_schema()}})
        return func
    return deco


class ToolError(Exception):
    def __init__(self, message: str, hint: str = ""):
        super().__init__(message)
        self.hint = hint


FAKE_WEATHER = {"北京": {"condition": "晴", "temp_c": 12}, "上海": {"condition": "小雨", "temp_c": 18}}


class WeatherArgs(BaseModel):
    city: str = Field(min_length=1, max_length=10, description="城市中文名，如 北京、上海")

    @field_validator("city")
    @classmethod
    def strip_city(cls, v: str) -> str:
        return v.strip().removesuffix("市")


@tool(WeatherArgs, "查询城市当前天气和气温。")
async def get_weather(city: str) -> dict:
    await asyncio.sleep(0.1)
    if city not in FAKE_WEATHER:
        raise ToolError(f"没有 {city} 的天气数据", hint=f"只支持：{', '.join(FAKE_WEATHER)}")
    return {"city": city, **FAKE_WEATHER[city]}


class CalcArgs(BaseModel):
    expression: str = Field(max_length=100, description="只含数字和 + - * / ** ( ) 的算术表达式")


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and (abs(right) > 100 or abs(left) > 10_000):
            raise ToolError("指数太大，拒绝计算", hint="底数绝对值 ≤ 10000、指数绝对值 ≤ 100")
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ToolError("表达式里有不允许的内容", hint="只能用数字和 + - * / ** ( )")


@tool(CalcArgs, "精确计算算术表达式，任何算数都用它。", timeout=1.0)
async def calculator(expression: str) -> dict:
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ToolError(f"表达式语法错误：{expression!r}", hint="检查括号是否配对")
    return {"expression": expression, "result": _eval(tree.body)}


NOTES_DIR = pathlib.Path("notes")


class ReadNoteArgs(BaseModel):
    filename: str = Field(description="notes 目录下的文件名，如 todo.md")


@tool(ReadNoteArgs, "读取本地 notes 目录下的笔记文件。不知道文件名时传 '*' 列出所有文件。")
async def read_note(filename: str) -> dict:
    files = sorted(p.name for p in NOTES_DIR.glob("*"))
    if filename == "*":
        return {"files": files}
    path = (NOTES_DIR / filename).resolve()
    if NOTES_DIR.resolve() not in path.parents:  # 防 ../../etc/passwd 这种越权路径
        raise ToolError("只能读 notes 目录里的文件", hint=f"可读文件：{files}")
    if not path.exists():
        raise ToolError(f"没有文件 {filename}", hint=f"可读文件：{files}")
    return {"filename": filename, "content": path.read_text(encoding="utf-8")[:2000]}


def err(kind, message, hint="", retryable=True) -> str:
    return json.dumps({"ok": False, "error_type": kind, "message": message,
                       "hint": hint, "retryable": retryable}, ensure_ascii=False)


async def run_tool(name: str, arguments: str) -> str:
    """第 12 天的执行器：永远返回字符串，绝不抛异常"""
    if name not in REGISTRY:
        return err("unknown_tool", f"没有叫 {name} 的工具", hint=f"可用工具：{', '.join(REGISTRY)}")
    args_model, func, timeout = REGISTRY[name]
    try:
        args = args_model.model_validate_json(arguments or "{}")
    except ValidationError as e:
        return err("invalid_arguments", str(e.errors()[0]["msg"]), hint="按 JSON Schema 重新传参")
    try:
        data = await asyncio.wait_for(func(**args.model_dump()), timeout)
        return json.dumps({"ok": True, "data": data}, ensure_ascii=False)
    except ToolError as e:
        return err("tool_error", str(e), hint=e.hint)
    except asyncio.TimeoutError:
        return err("timeout", f"{name} 超时", retryable=False)
    except Exception as e:
        return err("internal_error", f"{type(e).__name__}: {e}", retryable=False)


# ======================= llm.py：模型调用（流式） =======================
async def accumulate(chunks):
    """把流式 chunk 拼成：边拿到文字边 yield ("text", 片段)，最后 yield ("message", 完整 assistant 消息)。
    tool_calls 在流里是碎片：第一块带 id 和 name，后面每块只带一段 arguments，靠 index 归位。
    类比 Java：StringBuilder[] builders，按下标 append。"""
    text, calls = [], {}  # index -> {"id", "name", "arguments"}
    async for chunk in chunks:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        if delta.content:
            text.append(delta.content)
            yield "text", delta.content
        for tc in delta.tool_calls or []:
            c = calls.setdefault(tc.index, {"id": "", "name": "", "arguments": ""})
            c["id"] = tc.id or c["id"]
            if tc.function and tc.function.name:
                c["name"] += tc.function.name
            if tc.function and tc.function.arguments:
                c["arguments"] += tc.function.arguments
    msg = {"role": "assistant", "content": "".join(text) or None}
    if calls:
        msg["tool_calls"] = [{"id": c["id"], "type": "function",
                              "function": {"name": c["name"], "arguments": c["arguments"]}}
                             for _, c in sorted(calls.items())]
    yield "message", msg


async def real_stream(messages, tools):
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    stream = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"), messages=messages, tools=tools,
        temperature=0, stream=True)
    async for event in accumulate(stream):
        yield event


def _chunk(content=None, tool_calls=None):
    return NS(choices=[NS(delta=NS(content=content, tool_calls=tool_calls))])


def _tc_chunks(index, cid, name, args: dict):
    """模拟真实 API：一个工具调用拆成 1 块头 + 若干块 arguments 碎片"""
    raw = json.dumps(args, ensure_ascii=False)
    yield _chunk(tool_calls=[NS(index=index, id=cid, function=NS(name=name, arguments=""))])
    for i in range(0, len(raw), 4):
        yield _chunk(tool_calls=[NS(index=index, id=None, function=NS(name=None, arguments=raw[i:i + 4]))])


def _script(messages):
    """假模型的"大脑"：看最后一条 user 消息和已有的工具结果，决定下一步"""
    last_user = max(i for i, m in enumerate(messages) if m["role"] == "user")
    q = messages[last_user]["content"]
    results = {m["tool_call_id"]: json.loads(m["content"]) for m in messages[last_user:] if m["role"] == "tool"}
    if "北京" in q and "天气" in q:
        if not results:
            return [("t1", "get_weather", {"city": "北京"})]
        d = results["t1"]["data"]
        return f"北京现在{d['condition']}，{d['temp_c']}°C，出门可以穿件外套。"
    if "上海呢" in q:  # 没说"天气"，要靠多轮记忆知道问的是天气
        if not results:
            return [("t2", "get_weather", {"city": "上海"}), ("t3", "get_weather", {"city": "广州"})]
        return (f"上海{results['t2']['data']['condition']}，{results['t2']['data']['temp_c']}°C。"
                f"顺便查了广州，但{results['t3']['message']}（{results['t3']['hint']}）。")
    if "温差" in q:
        if not results:
            return [("t4", "calculator", {"expression": "18-12"})]
        return f"上海比北京高 {results['t4']['data']['result']}°C。"
    if "笔记" in q:
        if not results:
            return [("t5", "read_note", {"filename": "*"})]
        if "t6" not in results:
            return [("t6", "read_note", {"filename": results["t5"]["data"]["files"][0]})]
        return "你的待办：\n" + results["t6"]["data"]["content"].strip()
    return "我是假模型，只会按剧本回答 😄 填上 Key 试试真模型吧。"


async def fake_stream(messages, tools):
    async def chunks():
        plan = _script(messages)
        if isinstance(plan, str):
            for i in range(0, len(plan), 3):
                await asyncio.sleep(0.03)
                yield _chunk(content=plan[i:i + 3])
        else:
            for idx, (cid, name, args) in enumerate(plan):
                for c in _tc_chunks(idx, cid, name, args):
                    await asyncio.sleep(0.01)
                    yield c
    async for event in accumulate(chunks()):
        yield event


# ======================= main.py：对话循环 =======================
SYSTEM = ("你是 Jiawei 的命令行助手，回答简洁。天气、计算、读笔记必须调用工具。"
          "工具返回 ok=false 时读 hint 改正；retryable=false 就如实告诉用户。")
MAX_ROUNDS = 5
MAX_HISTORY = 20  # 只保留最近 20 条（不含 system），防止上下文无限增长


class ChatSession:
    def __init__(self, llm):
        self.llm = llm
        self.messages = [{"role": "system", "content": SYSTEM}]

    def _trim(self):
        rest = self.messages[1:]
        while len(rest) > MAX_HISTORY:
            rest.pop(0)
            while rest and rest[0]["role"] != "user":  # 不能从 tool 消息中间切断
                rest.pop(0)
        self.messages = self.messages[:1] + rest

    async def turn(self, user_input: str) -> None:
        self.messages.append({"role": "user", "content": user_input})
        self._trim()
        seen: dict[tuple, int] = {}
        for _ in range(MAX_ROUNDS):
            print("助手：", end="", flush=True)
            reply = None
            async for kind, value in self.llm(self.messages, TOOLS):
                if kind == "text":
                    print(value, end="", flush=True)  # 逐字输出
                else:
                    reply = value
            self.messages.append(reply)
            if not reply.get("tool_calls"):
                print()
                return
            print("（调用工具）")

            async def handle(c):
                key = (c["function"]["name"], c["function"]["arguments"])
                seen[key] = seen.get(key, 0) + 1
                if seen[key] > 2:
                    return err("repeated_call", "同样的调用已执行过", retryable=False)
                return await run_tool(*key)

            results = await asyncio.gather(*(handle(c) for c in reply["tool_calls"]))
            for c, r in zip(reply["tool_calls"], results):
                print(f"   🔧 {c['function']['name']}({c['function']['arguments']}) → {r}")
                self.messages.append({"role": "tool", "tool_call_id": c["id"], "content": r})
        print(f"\n⚠️ 超过 {MAX_ROUNDS} 轮，强制停止")


def prepare_notes():
    NOTES_DIR.mkdir(exist_ok=True)
    f = NOTES_DIR / "todo.md"
    if not f.exists():
        f.write_text("- 完成第 13–14 天阶段项目\n- 和老师细化第 2 阶段日程\n", encoding="utf-8")


async def main():
    prepare_notes()
    llm = real_stream if os.getenv("LLM_API_KEY") else fake_stream
    print("使用", "真模型" if llm is real_stream else "假模型（没填 Key）", "\n")
    session = ChatSession(llm)
    # 浏览器里没有 input()，用剧本代替；本地版换成 while True: input("你：")
    for q in ["北京天气怎么样？", "那上海呢？顺便看看广州", "两地温差多少？", "看看我的笔记里有什么待办"]:
        print(f"\n你：{q}")
        await session.turn(q)
    print(f"\n（本次会话共 {len(session.messages)} 条消息）")


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await